# Traditional RAG with ChromaDB + optional Pinecone

> **Teaching goal:** this notebook is designed as a step-by-step, instructor-friendly Colab lab. Every important cell explains what it does, why it matters, and how it fits into RAG / GraphRAG.

## Corpus used in this lab
We use public research papers as a realistic mini-corpus:

1. **Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks** — Lewis et al.
2. **From Local to Global: A Graph RAG Approach to Query-Focused Summarization** — Edge et al.

These papers are useful because they naturally contain entities, methods, datasets, claims, and relationships. That makes them good for comparing:

- **Traditional RAG:** retrieve semantically similar chunks.
- **GraphRAG:** retrieve facts and relationships from a knowledge graph.
- **Hybrid RAG:** combine vector retrieval and graph traversal.

## What this notebook demonstrates

This notebook implements **traditional vector RAG** using a file-based persistent vector database.

You will learn:

1. How to download research papers.
2. How to split PDFs into chunks.
3. How to embed chunks.
4. How to store embeddings in **ChromaDB** on disk.
5. How to ask questions using retrieved context.
6. How to optionally switch to **Pinecone** for hosted vector search.

Traditional RAG is best when the question can be answered by retrieving a few semantically relevant passages.

In [ ]:
# ============================================================
# 1. Install dependencies
# ============================================================
# In Colab, run this cell first. It installs:
# - openai: LLM + embeddings
# - pymupdf: PDF text extraction
# - pandas/numpy/tqdm: data handling
# - tiktoken: optional token-aware splitting

!pip -q install openai pymupdf pandas numpy tqdm tiktoken

!pip -q install chromadb pinecone

In [ ]:
# ============================================================
# 2. Configure API keys safely
# ============================================================
# Recommended in Colab:
#   Left sidebar -> Secrets -> add OPENAI_API_KEY
# Then this cell will read it without exposing the key in the notebook.

import os

try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY") or os.environ.get("OPENAI_API_KEY", "")
except Exception:
    # Not running in Colab. You can set your key in your environment instead.
    pass

if not os.environ.get("OPENAI_API_KEY"):
    raise ValueError("Please set OPENAI_API_KEY in Colab Secrets or as an environment variable.")

from openai import OpenAI
client = OpenAI()

CHAT_MODEL = "gpt-4o-mini"
EMBED_MODEL = "text-embedding-3-small"

print("OpenAI client configured. Models:", CHAT_MODEL, EMBED_MODEL)

In [ ]:
# ============================================================
# 3. Common helper functions: LLM + embeddings
# ============================================================
# llm() is used for answer generation, extraction, routing, and evaluation.
# embed_texts() converts a list of strings into embedding vectors.

import numpy as np

def llm(prompt, system="You are a precise, helpful assistant.", temperature=0.0):
    """Single-turn LLM call. Returns plain text."""
    response = client.chat.completions.create(
        model=CHAT_MODEL,
        temperature=temperature,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": prompt},
        ],
    )
    return response.choices[0].message.content.strip()


def embed_texts(texts, batch_size=64):
    """Embed a string or list of strings using OpenAI embeddings."""
    if isinstance(texts, str):
        texts = [texts]
    vectors = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        response = client.embeddings.create(model=EMBED_MODEL, input=batch)
        vectors.extend([d.embedding for d in response.data])
    return np.array(vectors, dtype="float32")

print(llm("Reply with exactly: ready"))
print("Embedding shape for one text:", embed_texts("hello").shape)

In [ ]:
# ============================================================
# 4. Download public research papers and extract text
# ============================================================
# We download PDFs from public URLs and extract text with PyMuPDF.
# The resulting `documents` list will be used by RAG, GraphRAG, and Hybrid RAG.

import requests, fitz, re
from pathlib import Path
from tqdm.auto import tqdm

DATA_DIR = Path("/content/rag_graphrag_corpus")
DATA_DIR.mkdir(parents=True, exist_ok=True)

PAPERS = [
    {
        "doc_id": "rag_lewis_2020",
        "title": "Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks",
        "url": "https://arxiv.org/pdf/2005.11401",
    },
    {
        "doc_id": "graphrag_edge_2024",
        "title": "From Local to Global: A Graph RAG Approach to Query-Focused Summarization",
        "url": "https://arxiv.org/pdf/2404.16130",
    },
]


def download_pdf(url, path):
    if path.exists() and path.stat().st_size > 10_000:
        return path
    r = requests.get(url, timeout=60)
    r.raise_for_status()
    path.write_bytes(r.content)
    return path


def extract_pdf_text(path):
    doc = fitz.open(path)
    pages = []
    for page_num, page in enumerate(doc, start=1):
        text = page.get_text("text")
        text = re.sub(r"\s+", " ", text).strip()
        if text:
            pages.append({"page": page_num, "text": text})
    return pages

# Build a page-level document list.
documents = []
for paper in tqdm(PAPERS):
    pdf_path = DATA_DIR / f"{paper['doc_id']}.pdf"
    download_pdf(paper["url"], pdf_path)
    for page in extract_pdf_text(pdf_path):
        documents.append({
            "doc_id": paper["doc_id"],
            "title": paper["title"],
            "page": page["page"],
            "text": page["text"],
        })

print(f"Loaded {len(documents)} page-documents")
print(documents[0].keys())
print(documents[0]["title"], "page", documents[0]["page"])
print(documents[0]["text"][:500])

In [ ]:
# ============================================================
# 5. Chunk the corpus
# ============================================================
# Why chunk?
# LLMs and vector databases work better with small passages than entire PDFs.
# Each chunk keeps metadata so we can cite the source paper and page.

import textwrap


def chunk_text(text, chunk_size=900, overlap=150):
    """Simple character-based chunking with overlap."""
    chunks = []
    start = 0
    while start < len(text):
        end = min(start + chunk_size, len(text))
        chunk = text[start:end].strip()
        if len(chunk) > 100:
            chunks.append(chunk)
        start += chunk_size - overlap
    return chunks

chunks = []
for d in documents:
    for j, chunk in enumerate(chunk_text(d["text"])):
        chunks.append({
            "id": f"{d['doc_id']}_p{d['page']}_c{j}",
            "text": chunk,
            "doc_id": d["doc_id"],
            "title": d["title"],
            "page": d["page"],
            "chunk_id": j,
        })

print(f"Created {len(chunks)} chunks")
print(chunks[0]["id"])
print(textwrap.shorten(chunks[0]["text"], width=600))

## Build a file-based ChromaDB vector store

ChromaDB can run locally in Colab and persist data to a folder. That means the vector database is stored on Colab's file system, not only in a Python variable.

In this section:

- `PersistentClient(path=...)` creates a file-backed database.
- `collection.add(...)` stores chunks, metadata, and embeddings.
- The vector DB can later return the most similar chunks for a question.

In [ ]:
# ============================================================
# 6. Create a persistent ChromaDB collection
# ============================================================
# Chroma stores:
# - documents: the actual chunk text
# - embeddings: numeric vectors
# - metadatas: title/page/source info
# - ids: stable unique chunk IDs

import chromadb
from chromadb.config import Settings

CHROMA_DIR = "/content/chroma_research_rag"
chroma_client = chromadb.PersistentClient(path=CHROMA_DIR)

collection = chroma_client.get_or_create_collection(
    name="research_papers_rag",
    metadata={"description": "RAG and GraphRAG research paper chunks"},
)

print("Collection ready:", collection.name)
print("Existing rows:", collection.count())

In [ ]:
# ============================================================
# 7. Embed chunks and insert them into Chroma
# ============================================================
# For teaching, we rebuild only if the collection is empty.
# Chroma persists under CHROMA_DIR, so rerunning this cell does not need to duplicate rows.

if collection.count() == 0:
    texts = [c["text"] for c in chunks]
    ids = [c["id"] for c in chunks]
    metadatas = [
        {
            "doc_id": c["doc_id"],
            "title": c["title"],
            "page": c["page"],
            "chunk_id": c["chunk_id"],
        }
        for c in chunks
    ]

    print("Embedding chunks...")
    embeddings = embed_texts(texts).tolist()

    print("Adding to Chroma...")
    # Add in small batches to avoid large request payloads.
    batch_size = 100
    for i in tqdm(range(0, len(chunks), batch_size)):
        collection.add(
            ids=ids[i:i+batch_size],
            documents=texts[i:i+batch_size],
            metadatas=metadatas[i:i+batch_size],
            embeddings=embeddings[i:i+batch_size],
        )

print("Chroma rows:", collection.count())
print("Persisted at:", CHROMA_DIR)

## Retrieve relevant chunks

A vector DB search has three steps:

1. Embed the user question.
2. Search for nearest chunk vectors.
3. Return chunk text and metadata.

The returned passages become the **context** for the LLM.

In [ ]:
# ============================================================
# 8. Retrieval function
# ============================================================

def chroma_retrieve(query, k=5):
    q_emb = embed_texts(query)[0].tolist()
    result = collection.query(
        query_embeddings=[q_emb],
        n_results=k,
        include=["documents", "metadatas", "distances"],
    )

    hits = []
    for doc, meta, distance in zip(
        result["documents"][0],
        result["metadatas"][0],
        result["distances"][0],
    ):
        hits.append({"text": doc, "metadata": meta, "distance": distance})
    return hits

query = "What problem does Retrieval-Augmented Generation solve?"
hits = chroma_retrieve(query, k=3)

for i, h in enumerate(hits, start=1):
    m = h["metadata"]
    print(f"\n--- Hit {i} | {m['doc_id']} page {m['page']} | distance={h['distance']:.4f} ---")
    print(textwrap.shorten(h["text"], width=700))

## Generate an answer from retrieved context

This is the **G** in RAG: generation.

The prompt intentionally says **use only the context**. This reduces hallucination because the model is asked to ground its answer in retrieved passages.

In [ ]:
# ============================================================
# 9. Traditional RAG answer function
# ============================================================

def format_hits_for_prompt(hits):
    blocks = []
    for i, h in enumerate(hits, start=1):
        m = h["metadata"]
        blocks.append(
            f"[Source {i}: {m['title']} | page {m['page']} | chunk {m['chunk_id']}]\n{h['text']}"
        )
    return "\n\n".join(blocks)


def traditional_rag_answer(query, k=5, verbose=True):
    hits = chroma_retrieve(query, k=k)
    context = format_hits_for_prompt(hits)
    prompt = f"""
Answer the question using ONLY the context below.
If the answer is not fully supported by the context, say what is missing.
Cite sources inline as [Source 1], [Source 2], etc.

Context:
{context}

Question: {query}
Answer:
"""
    answer = llm(prompt)

    if verbose:
        print("Retrieved sources:")
        for i, h in enumerate(hits, start=1):
            m = h["metadata"]
            print(f"  Source {i}: {m['doc_id']} page {m['page']} distance={h['distance']:.4f}")
        print("\nAnswer:\n", answer)

    return answer, hits

_ = traditional_rag_answer("How does RAG combine parametric and non-parametric memory?", k=5)

## Try real use-case questions

These examples show where traditional RAG works well:

- definition questions
- method summary questions
- finding a passage about a specific concept
- comparing nearby concepts when the answer exists in local chunks

In [ ]:
QUESTIONS = [
    "What is Retrieval-Augmented Generation?",
    "Why can naive vector RAG struggle with global questions over an entire corpus?",
    "What does the GraphRAG paper propose as an alternative to naive RAG?",
]

for q in QUESTIONS:
    print("\n" + "="*100)
    print("QUESTION:", q)
    traditional_rag_answer(q, k=5, verbose=True)

## Optional: Pinecone version

Use this section when you want a hosted vector database instead of local Chroma.

You need:

- `PINECONE_API_KEY`
- a Pinecone account

The retrieval/generation logic stays almost the same. Only the vector-store backend changes.

In [ ]:
# ============================================================
# 10. Optional Pinecone setup
# ============================================================
# This cell is intentionally optional. It will skip if PINECONE_API_KEY is not set.

try:
    from pinecone import Pinecone, ServerlessSpec

    PINECONE_API_KEY = os.environ.get("PINECONE_API_KEY", "")
    if not PINECONE_API_KEY:
        print("Skipping Pinecone: set PINECONE_API_KEY to run this section.")
    else:
        pc = Pinecone(api_key=PINECONE_API_KEY)
        index_name = "research-rag-demo"
        dim = embed_texts("dimension check").shape[1]

        existing = [idx["name"] for idx in pc.list_indexes()]
        if index_name not in existing:
            pc.create_index(
                name=index_name,
                dimension=dim,
                metric="cosine",
                spec=ServerlessSpec(cloud="aws", region="us-east-1"),
            )

        pinecone_index = pc.Index(index_name)
        print("Pinecone index ready:", index_name, "dim=", dim)
except Exception as e:
    print("Pinecone section skipped or failed:", e)

In [ ]:
# ============================================================
# 11. Optional Pinecone upsert and retrieval
# ============================================================
# Run only if the previous Pinecone cell created `pinecone_index`.

if "pinecone_index" in globals():
    vectors = embed_texts([c["text"] for c in chunks]).tolist()
    batch = []
    for c, v in zip(chunks, vectors):
        batch.append({
            "id": c["id"],
            "values": v,
            "metadata": {
                "text": c["text"][:3000],
                "doc_id": c["doc_id"],
                "title": c["title"],
                "page": c["page"],
                "chunk_id": c["chunk_id"],
            },
        })

    for i in tqdm(range(0, len(batch), 100)):
        pinecone_index.upsert(vectors=batch[i:i+100])

    def pinecone_retrieve(query, k=5):
        qv = embed_texts(query)[0].tolist()
        res = pinecone_index.query(vector=qv, top_k=k, include_metadata=True)
        return res["matches"]

    matches = pinecone_retrieve("What is GraphRAG?", k=3)
    for m in matches:
        print(m["score"], m["metadata"]["doc_id"], m["metadata"]["page"])
else:
    print("Pinecone index not configured. This is okay for the Chroma-based lab.")

## Teaching recap

Traditional RAG workflow:

```text
PDFs → text extraction → chunks → embeddings → vector DB → top-k chunks → LLM answer
```

Main advantage:

- Fast semantic lookup over large text collections.

Main limitation:

- It may struggle with multi-hop relationship questions or global corpus-level questions where no single chunk is enough.